# Supervisor Explorer V2

Interactive notebook for running and visualizing the managed fabrication pipeline.

**Sections:**
1. Setup
2. Run Prompt Job (iter_000)
3. Inspect Iteration Artifacts
4. Apply Deterministic Redesign (dot-path edits)
5. Apply Parameter Overrides
6. Metrics Dashboard (history + delta)
7. 3D Primitive Viewer
8. GWL Layer Preview
9. Smoke Test (automated verification)

---
## 1. Setup

In [ ]:
import sys
import json
import copy
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import numpy as np

# -------------------------------------------------------
# Path bootstrap
# -------------------------------------------------------
NOTEBOOK_DIR = Path().resolve()
SRC_DIR      = NOTEBOOK_DIR / 'src'
SUPER_DIR    = SRC_DIR / 'supervisor'
OUTPUTS_DIR  = NOTEBOOK_DIR / 'Outputs'

for p in [str(SRC_DIR), str(SUPER_DIR), str(NOTEBOOK_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from fabrication_job import FabricationJob
from supervisor import Supervisor
from dot_path_editor import apply_dot_path_edits, validate_edit_dict

print('Imports OK')
print(f'NOTEBOOK_DIR : {NOTEBOOK_DIR}')
print(f'OUTPUTS_DIR  : {OUTPUTS_DIR}')

In [ ]:
# -------------------------------------------------------
# Instantiate the Supervisor
# -------------------------------------------------------
sup = Supervisor(
    outputs_root=OUTPUTS_DIR,
    # param_file is auto-discovered from PrintParameters.txt
)
print(sup)

---
## 2. Run Prompt Job (iter_000)

Edit the `JOB_ID` and `PROMPT` below, then run the cell.

In [ ]:
# -------------------------------------------------------
# Configure your job
# -------------------------------------------------------
JOB_ID = 'pyramid_grid_demo'
PROMPT = (
    'Design a 5x5 grid of sharp pyramids. '
    'Each pyramid has base_width_um=10, height_um=20, layers=20. '
    'Grid spacing 25 um in X and Y.'
)

job = FabricationJob(job_id=JOB_ID, base_prompt=PROMPT)
print(job.summary())

In [ ]:
# -------------------------------------------------------
# Run iter_000 (no overrides)
# -------------------------------------------------------
sup.run_prompt_job(job)
print(f'\nJob complete. Iteration now at: {job.iteration}')
print(f'Metrics: {job.metrics_history[-1]}')

---
## 3. Inspect Iteration Artifacts

In [ ]:
def load_iter_artifacts(job: FabricationJob, iteration: int, outputs_root: Path) -> dict:
    """Load all JSON artifacts for a given iteration."""
    iter_dir = outputs_root / job.job_id / f'iter_{iteration:03d}'
    result = {'iter_dir': iter_dir}
    for fname in ('design.json', 'reduced.json', 'enriched_reduced.json',
                  'output.json', 'metrics.json', 'feedback_log.json'):
        p = iter_dir / fname
        if p.exists():
            with open(p) as f:
                result[fname] = json.load(f)
        else:
            result[fname] = None
    return result


# Inspect iter_000
iter0 = load_iter_artifacts(job, 0, OUTPUTS_DIR)

print('=== iter_000 metrics ===')
print(json.dumps(iter0.get('metrics.json'), indent=2))

print('\n=== iter_000 design (objects) ===')
d = iter0.get('design.json') or {}
for obj_name, obj_def in (d.get('objects') or {}).items():
    print(f'  {obj_name}: {obj_def.get("type")} | ', end='')
    if obj_def.get('type') == 'geometry':
        print(f'{len(obj_def.get("components", []))} component(s)')
    else:
        print(f'uses={obj_def.get("uses")} repeat={obj_def.get("repeat")}')

print(f'\n=== Assembly ===')
print(json.dumps(d.get('assembly'), indent=2))

In [ ]:
# Show first 3 primitives from enriched_reduced
enriched = iter0.get('enriched_reduced.json') or {}
prims = enriched.get('primitives', [])
print(f'Total primitives: {len(prims)}')
print('\n--- First 3 primitives ---')
for p in prims[:3]:
    print(json.dumps(p, indent=2))

---
## 4. Apply Deterministic Redesign (dot-path edits)

Edit `REDESIGN_EDITS` and run. This loads iter_000 design from disk, applies edits, re-validates, and re-runs the pipeline producing iter_001.

In [ ]:
# -------------------------------------------------------
# Define dot-path edits
# Keys are dot-separated paths into the design JSON.
# -------------------------------------------------------
REDESIGN_EDITS = {
    # Example: change assembly grid size
    'assembly.grid.x': 7,
    'assembly.grid.y': 7,
}

# Pre-flight validation
errors = validate_edit_dict(REDESIGN_EDITS)
if errors:
    print('Edit validation errors:')
    for e in errors:
        print(f'  {e}')
else:
    print('Edit dict is valid. Ready to apply.')
    print(json.dumps(REDESIGN_EDITS, indent=2))

In [ ]:
# -------------------------------------------------------
# Apply redesign -> produces iter_001
# -------------------------------------------------------
sup.apply_redesign(job, edits=REDESIGN_EDITS)
print(f'\nRedesign complete. Iteration now at: {job.iteration}')
print(f'Metrics: {job.metrics_history[-1]}')

In [ ]:
# Verify the edit was applied on disk
iter1 = load_iter_artifacts(job, 1, OUTPUTS_DIR)
design1 = iter1.get('design.json') or {}
print('iter_001 assembly:')
print(json.dumps(design1.get('assembly'), indent=2))

---
## 5. Apply Parameter Overrides

Overrides are applied AFTER the manufacturing agent (post-MA, pre-endpoint).

- **Delta**: string like `"+0.1"` or `"-0.05"` -> added to existing value
- **Absolute**: numeric -> replaces existing value

In [ ]:
# -------------------------------------------------------
# Define overrides
# -------------------------------------------------------
OVERRIDES = {
    'slice_spacing_um': '+0.05',   # delta: increase by 0.05 um
    'hatch_spacing_um': '-0.03',   # delta: decrease by 0.03 um
}

# This re-uses iter_001's enriched data; produces iter_002 with adjusted params
sup.apply_overrides(job, overrides=OVERRIDES)
print(f'\nOverride iteration complete. Iteration now at: {job.iteration}')
print(f'Metrics: {job.metrics_history[-1]}')

In [ ]:
# Inspect overridden enriched data
iter2 = load_iter_artifacts(job, 2, OUTPUTS_DIR)
enriched2 = iter2.get('enriched_reduced.json') or {}
p0 = enriched2.get('primitives', [{}])[0]
print('First primitive local_parameters after overrides:')
print(json.dumps(p0.get('local_parameters'), indent=2))

---
## 6. Metrics Dashboard

In [ ]:
def plot_metrics_history(job: FabricationJob):
    """Bar charts for key metrics across iterations + delta table."""
    if not job.metrics_history:
        print('No metrics yet.')
        return

    history = job.metrics_history
    iters = list(range(len(history)))

    KEYS = [
        'total_primitives', 'total_layers', 'max_z',
        'average_risk', 'max_risk', 'mean_slenderness',
        'estimated_endpoint_count',
    ]

    n_keys = len(KEYS)
    n_cols = 4
    n_rows = (n_keys + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
    fig.suptitle(f'Iteration Metrics | Job: {job.job_id}', fontsize=14, fontweight='bold')
    axes = axes.flatten()

    colors = plt.cm.tab10(np.linspace(0, 1, len(iters)))

    for ax_idx, key in enumerate(KEYS):
        vals = [h.get(key, 0) for h in history]
        axes[ax_idx].bar(iters, vals, color=colors[:len(iters)])
        axes[ax_idx].set_title(key.replace('_', ' '), fontsize=9)
        axes[ax_idx].set_xlabel('Iteration')
        axes[ax_idx].set_xticks(iters)
        axes[ax_idx].tick_params(axis='x', labelsize=8)

    # Hide unused subplots
    for ax_idx in range(len(KEYS), len(axes)):
        axes[ax_idx].set_visible(False)

    plt.tight_layout()
    plt.show()

    # Delta table
    print('\n=== Delta Table ===')
    header = f'{'Metric':<35}' + ''.join(f'iter_{i:03d}  ' for i in iters)
    print(header)
    print('-' * len(header))
    for key in KEYS:
        row = f'{key:<35}'
        for h in history:
            val = h.get(key, 0)
            row += f'{val:<9.4g}  '
        print(row)

    print('\n=== Delta vs Previous Iteration ===')
    print(f'{'Metric':<35}' + ''.join(f'iter_{i:03d}  ' for i in iters))
    print('-' * len(header))
    for key in KEYS:
        row = f'{key:<35}'
        for h in history:
            delta = h.get('delta', {}).get(key, None)
            if delta is None:
                row += f'{'N/A':<11}'
            else:
                sign = '+' if delta >= 0 else ''
                row += f'{sign}{delta:<9.4g}  '
        print(row)


plot_metrics_history(job)

---
## 7. 3D Primitive Viewer

Lightweight scatter / box plot of primitive centers colored by risk.

In [ ]:
def plot_primitives_3d(job: FabricationJob, iteration: int, outputs_root: Path):
    """Scatter plot of primitive centers colored by risk score."""
    arts = load_iter_artifacts(job, iteration, outputs_root)
    enriched = arts.get('enriched_reduced.json') or {}
    primitives = enriched.get('primitives', [])

    if not primitives:
        print(f'No primitives found for iter_{iteration:03d}')
        return

    xs = [p['center'][0] for p in primitives]
    ys = [p['center'][1] for p in primitives]
    zs = [p['center'][2] for p in primitives]
    risks = [p.get('risk', 0.0) for p in primitives]

    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')

    sc = ax.scatter(xs, ys, zs, c=risks, cmap='RdYlGn_r',
                    s=15, alpha=0.6, vmin=0, vmax=1)

    cbar = fig.colorbar(sc, ax=ax, pad=0.12, shrink=0.6)
    cbar.set_label('Risk Score', fontsize=9)

    ax.set_xlabel('X (um)', fontsize=8)
    ax.set_ylabel('Y (um)', fontsize=8)
    ax.set_zlabel('Z (um)', fontsize=8)
    ax.set_title(
        f'Primitives | {job.job_id} | iter_{iteration:03d}\n'
        f'n={len(primitives)} | mean_risk={np.mean(risks):.3f}',
        fontsize=10
    )

    plt.tight_layout()
    plt.show()
    print(f'Plotted {len(primitives)} primitives.')


# Show all completed iterations
for it in range(job.iteration):
    plot_primitives_3d(job, it, OUTPUTS_DIR)

---
## 8. GWL Layer Preview

Visualize the first N layers of the endpoint JSON as scan-path segments.

In [ ]:
def plot_gwl_layers(job: FabricationJob, iteration: int, outputs_root: Path, n_layers: int = 6):
    """Plot first n_layers of output.json as scan paths."""
    arts = load_iter_artifacts(job, iteration, outputs_root)
    output = arts.get('output.json') or {}
    layers = output.get('layers', [])

    if not layers:
        print('No layers found.')
        return

    layers_to_show = layers[:n_layers]
    n = len(layers_to_show)
    n_cols = min(n, 3)
    n_rows = (n + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    fig.suptitle(
        f'GWL Layer Preview | {job.job_id} | iter_{iteration:03d} | '
        f'first {n} of {len(layers)} layers',
        fontsize=11, fontweight='bold'
    )

    if n == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if n > 1 else [axes]

    for ax_idx, layer in enumerate(layers_to_show):
        ax = axes[ax_idx]
        z = layer.get('z_um', 0)
        segs = layer.get('segments', [])

        hatch_segs   = [s for s in segs if s.get('type') == 'hatch']
        contour_segs = [s for s in segs if s.get('type') == 'contour']

        for s in hatch_segs:
            ax.plot(
                [s['start'][0], s['end'][0]],
                [s['start'][1], s['end'][1]],
                color='steelblue', lw=0.5, alpha=0.7
            )
        for s in contour_segs:
            ax.plot(
                [s['start'][0], s['end'][0]],
                [s['start'][1], s['end'][1]],
                color='tomato', lw=1.0, alpha=0.9
            )

        ax.set_title(f'Z = {z:.3f} um\n{len(segs)} segments', fontsize=8)
        ax.set_aspect('equal')
        ax.tick_params(labelsize=7)
        ax.set_xlabel('X (um)', fontsize=7)
        ax.set_ylabel('Y (um)', fontsize=7)

    handles = [
        mpatches.Patch(color='steelblue', label='Hatch'),
        mpatches.Patch(color='tomato',    label='Contour'),
    ]
    fig.legend(handles=handles, loc='lower right', fontsize=9)

    for ax_idx in range(n, len(axes)):
        axes[ax_idx].set_visible(False)

    plt.tight_layout()
    plt.show()


# Preview latest iteration
plot_gwl_layers(job, job.iteration - 1, OUTPUTS_DIR, n_layers=6)

---
## 9. Smoke Test (Automated Verification)

Runs a minimal self-contained test to verify all supervisor components work correctly.
Safe to run after cells 1-8 have already executed.

In [ ]:
import traceback

SMOKE_JOB_ID = '_smoke_test_'
SMOKE_PROMPT = (
    'Design a simple 3x3 grid of box pillars. '
    'Each pillar: x_um=5, y_um=5, z_um=10. '
    'Grid spacing 15 um in X and Y.'
)

failures = []

# ---- Test 1: run_prompt_job produces iter_000 ----
try:
    smoke_job = FabricationJob(job_id=SMOKE_JOB_ID, base_prompt=SMOKE_PROMPT)
    smoke_sup = Supervisor(outputs_root=OUTPUTS_DIR)
    smoke_sup.run_prompt_job(smoke_job)

    iter0_dir = OUTPUTS_DIR / SMOKE_JOB_ID / 'iter_000'
    required_files = [
        'design.json', 'reduced.json', 'enriched_reduced.json',
        'output.json', 'metrics.json', 'feedback_log.json',
    ]
    for fname in required_files:
        assert (iter0_dir / fname).exists(), f'MISSING: {fname}'
    assert (iter0_dir / 'gwl').is_dir(), 'MISSING: gwl/'
    assert smoke_job.iteration == 1
    assert len(smoke_job.metrics_history) == 1
    print('[PASS] Test 1: run_prompt_job produces iter_000 with all artifacts')
except Exception as e:
    failures.append(f'Test 1 FAILED: {e}')
    print(f'[FAIL] Test 1: {e}')
    traceback.print_exc()

# ---- Test 2: apply_redesign produces iter_001 with modified value ----
SMOKE_EDITS = {'assembly.grid.x': 4, 'assembly.grid.y': 4}
try:
    smoke_sup.apply_redesign(smoke_job, edits=SMOKE_EDITS)

    iter1_dir = OUTPUTS_DIR / SMOKE_JOB_ID / 'iter_001'
    assert iter1_dir.exists(), 'iter_001 not created'
    with open(iter1_dir / 'design.json') as f:
        d1 = json.load(f)
    assert d1['assembly']['grid']['x'] == 4, 'assembly.grid.x not updated'
    assert d1['assembly']['grid']['y'] == 4, 'assembly.grid.y not updated'
    assert smoke_job.iteration == 2
    assert len(smoke_job.metrics_history) == 2
    assert 'delta' in smoke_job.metrics_history[1], 'delta key missing'
    print('[PASS] Test 2: apply_redesign produces iter_001 with correct edits + delta')
except Exception as e:
    failures.append(f'Test 2 FAILED: {e}')
    print(f'[FAIL] Test 2: {e}')
    traceback.print_exc()

# ---- Test 3: apply_overrides produces iter_002 with changed local params ----
SMOKE_OVERRIDES = {'slice_spacing_um': '+0.05'}
try:
    smoke_sup.apply_overrides(smoke_job, overrides=SMOKE_OVERRIDES)

    iter2_dir = OUTPUTS_DIR / SMOKE_JOB_ID / 'iter_002'
    assert iter2_dir.exists(), 'iter_002 not created'
    with open(iter2_dir / 'enriched_reduced.json') as f:
        e2 = json.load(f)
    assert e2['primitives'], 'No primitives'
    assert smoke_job.iteration == 3
    assert len(smoke_job.feedback_log) > 0, 'feedback_log empty'
    print('[PASS] Test 3: apply_overrides produces iter_002 with logged overrides')
except Exception as e:
    failures.append(f'Test 3 FAILED: {e}')
    print(f'[FAIL] Test 3: {e}')
    traceback.print_exc()

# ---- Summary ----
print('\n' + '=' * 60)
if failures:
    print(f'SMOKE TEST: {len(failures)} FAILURE(S)')
    for f in failures:
        print(f'  {f}')
else:
    print('SMOKE TEST: ALL PASSED')
print('=' * 60)